In [23]:
import pandas as pd
import os

# Set project directory
project_dir = '/Users/omkar/Documents/SmartShop'
os.chdir(project_dir)

# Verify current working directory
print(f"Current directory: {os.getcwd()}")


Current directory: /Users/omkar/Documents/SmartShop


In [65]:
#### Data loader
Column_name = ['supermarket',"prices_(£)",'names','date','category','own_brand']
sains_data = pd.read_csv('data/All_Data_Sains.csv', usecols=Column_name)
#### Reading the brand - product csv
brand_df = pd.read_csv("/Users/omkar/Documents/SmartShop/notebook/llm_master_brand.csv",header=0,names=['names','brand'])


In [25]:
#### Data Preprocessing
import re 
## Changing the Date format.
def change_date_format(df:pd.DataFrame) -> pd.DataFrame:

    if "date" in df.columns:
        print("Date Column Processing")
        df['date'] = pd.to_datetime(df['date'], format='%Y%m%d')
        print("Date Column Processed")
    else:
        return None
    return df


## Extracting the packet size from the name column and creating a new column for it.

def extract_package_size(df:pd.DataFrame) -> pd.DataFrame:
    df['extracted_pack_info'] = df['names'].str.split().apply(lambda words: [w for w in words if re.search(r'\d', w)])
    return df


## Add a new column "Brand" and replace the supermarket name with the brand name with own_brand == True.
def replace_house_brand(df:pd.DataFrame)->pd.DataFrame:
    df.loc[df['own_brand'] == True, 'Brand'] = df['supermarket']
    return df


In [26]:
import re
import pandas as pd

def normalize_pack_info(df, col="extracted_pack_info"):
    
    def parse_pack(x):
        # Handle None / NaN safely
        if x is None:
            return None, None, None
        
        if isinstance(x, float) and pd.isna(x):
            return None, None, None
        
        # Convert list → string
        if isinstance(x, (list, tuple)):
            x = " ".join(map(str, x))
        else:
            x = str(x)
        
        x = x.lower()
        x = re.sub(r"[^\w\.x ]", " ", x)  # clean junk
        
        pack_count = None
        unit_size = None
        unit_type = None
        
        # Case 1: 4x85g
        match = re.search(r"(\d+)\s*x\s*(\d+)(g|ml|kg|l)", x)
        if match:
            return int(match.group(1)), int(match.group(2)), match.group(3)
        
        # Case 2: size + count
        size_match = re.search(r"(\d+)(g|ml|kg|l)", x)
        count_match = re.findall(r"\b\d+\b", x)
        
        if size_match:
            unit_size = int(size_match.group(1))
            unit_type = size_match.group(2)
        
        # Extract count carefully
        if count_match:
            nums = list(map(int, count_match))
            
            # remove unit_size if present
            if unit_size in nums:
                nums.remove(unit_size)
            
            if nums:
                pack_count = nums[0]
        
        # Case 3: x30
        if pack_count is None:
            match = re.search(r"x(\d+)", x)
            if match:
                pack_count = int(match.group(1))
        
        return pack_count, unit_size, unit_type
    
    parsed = df[col].apply(parse_pack)
    
    df["pack_count"] = parsed.apply(lambda x: x[0])
    df["unit_size"] = parsed.apply(lambda x: x[1])
    df["unit_type"] = parsed.apply(lambda x: x[2])
    
    return df

In [27]:
transform1 = change_date_format(sains_data)
print("Transformation Phase 1 Completed ")
transform2 = extract_package_size(transform1)
print("Transformation Phase 2 Completed ")
transform3 = replace_house_brand(transform2)
print("Transformation Phase 3 Completed ")
transform4 = normalize_pack_info(transform3)
print("Transformation Phase 4 Completed ")


Date Column Processing
Date Column Processed
Transformation Phase 1 Completed 
Transformation Phase 2 Completed 
Transformation Phase 3 Completed 
Transformation Phase 4 Completed 


In [28]:
transform4.head()

,supermarket,prices_(£),names,date,category,own_brand,extracted_pack_info,Brand,pack_count,unit_size,unit_type
0,Sains,6.00,Bassetts Vitamins Omega-3 & Multivits 3-6 Soft...,2024-04-13,baby_products,False,"[Omega-3, 3-6, x30]",NaN,3.0,NaN,NaN
1,Sains,2.35,Annabel Karmel Chicken & Potato Pie Toddler Me...,2024-04-13,baby_products,False,"[200g, 12]",NaN,12.0,200.0,g
2,Sains,2.35,Annabel Karmel Beef Cottage Pie Toddler Meal 2...,2024-04-13,baby_products,False,"[200g, 12]",NaN,12.0,200.0,g
3,Sains,2.15,Yeo Valley Organic Little Yeos 4x85g,2024-04-13,baby_products,False,[4x85g],NaN,4.0,85.0,g
4,Sains,0.75,The Collective Suckies Strawberry Kids Yoghurt...,2024-04-13,baby_products,False,[90g],NaN,NaN,90.0,g


In [7]:
final_dataset_columns = ['supermarket','price','product','date','category','Brand','pack_count','unit_size','unit_type']

In [29]:
#### Creating a Subset of the dataframe based on the brands. 
#### 2 subset will be created. inhouse-brand set and 3rd party set. 

inhouse_products = transform4[transform4['own_brand'] == True]
third_party_brands = transform4[transform4['own_brand'] == False]


### Brand Extraction

In [30]:
def normalize_brands(brands_list):
    import pandas as pd
    import re
    
    clean_map = {}
    
    for b in brands_list:
        if pd.isna(b):
            continue
        
        original = str(b).strip()
        
        # normalize
        norm = original.lower()
        norm = re.sub(r"[’']", "", norm)     # remove apostrophes
        norm = re.sub(r"[^a-z0-9\s]", " ", norm)  # remove special chars
        norm = re.sub(r"\s+", " ", norm).strip()  # remove extra spaces
        
        # keep first clean version as canonical
        if norm not in clean_map:
            clean_map[norm] = original
    
    return clean_map

In [31]:
brand_map = normalize_brands(brand_df['brand'].unique())
brand_map

{'maryland': 'Maryland',
 'weetabix': 'Weetabix',
 'walkers': "Walker's",
 'stamford street co': 'Stamford Street Co.',
 'cravendale': 'Cravendale',
 'js': 'JS',
 'heinz': 'Heinz',
 'napolina': 'Napolina',
 'oatly': 'Oatly',
 'alpro': 'Alpro',
 'ktc': 'KTC',
 'mcvities': "McVitie's",
 'princes': 'Princes',
 'hula hoops': 'Hula Hoops',
 'jacobs': "Jacob's",
 'batchelors': 'Batchelors',
 'pom bear': 'Pom-Bear',
 'birds eye': 'Birds Eye',
 'kitkat': 'KitKat',
 'diet coke': 'Diet Coke',
 'cadbury': 'Cadbury',
 'tilda': 'Tilda',
 'kit kat': 'Kit Kat',
 'ambrosia': 'Ambrosia',
 'pringles': 'Pringles',
 'kerrygold': 'Kerrygold',
 'doritos': 'Doritos',
 'sensations': 'Sensations',
 'branston': 'Branston',
 'jus rol': 'Jus-Rol',
 'happy egg': 'Happy Egg',
 'terrys': "Terry's",
 'kettle chips': 'Kettle Chips',
 'dolmio': 'Dolmio',
 'yeo valley': 'Yeo Valley',
 'mccoys': "McCoy's",
 'soreen': 'Soreen',
 'oreo': 'Oreo',
 'tunnocks': "Tunnock's",
 'quaker': 'Quaker',
 'lindt': 'Lindt',
 'bisto': 'B

In [32]:
brand_df.head(4)

,product,brand
0,Maryland Cookies Chocolate Chip Minis x6,Maryland
1,Weetabix Cereal x24,Weetabix
2,Walker's Shortbread Fingers x10 160g,Walker's
3,Stamford Street Co. Chopped Tomatoes in Tomato...,Stamford Street Co.


In [33]:
def map_brands_normalized(df, brand_map, text_col="names"):
    import re
    
    df = df.copy()
    
    # normalize product names
    names = df[text_col].fillna("").str.lower()
    names = names.str.replace(r"[’']", "", regex=True)
    names = names.str.replace(r"[^a-z0-9\s]", " ", regex=True)
    names = names.str.replace(r"\s+", " ", regex=True).str.strip()
    
    # sort normalized brands
    brands = sorted(brand_map.keys(), key=len, reverse=True)
    
    pattern = r'\b(' + '|'.join(map(re.escape, brands)) + r')\b'
    
    # extract normalized match
    df["Brand_norm"] = names.str.extract(pattern, expand=False)
    
    # map back to original brand
    df["Brand"] = df["Brand_norm"].map(brand_map)
    
    # fallback
    df["Brand"] = df["Brand"].fillna("Unknown")
    
    return df.drop(columns=["Brand_norm"])

In [34]:
mapped_brand_list = map_brands_normalized(third_party_brands, brand_map, text_col="names")

In [40]:
mapped_brand_list['Brand'].value_counts()

Brand
Unknown                1164460
L'Oreal Paris            26323
Cadbury                  16817
Extra                    16670
Stamford Street Co.      15939
                        ...   
Special K                   19
Koikeya                     18
Chef                        16
J-Basket                    10
Baking Buddy                 8
Name: count, Length: 985, dtype: int64

##### Brand List

In [67]:
brand_df['brand'].value_counts()

brand
Cadbury          232
Heinz            144
Walkers           88
Kellogg's         88
Twinings          81
                ... 
Kitkat             1
Oggs               1
Ben & Jerry's      1
Pulsin             1
J-Basket           1
Name: count, Length: 1046, dtype: int64